# P1 — Severity-Stratified LoRA Routing

**Architecture:** Frozen B2 WavLM encoder + 4 severity-specific LoRA adapters + re-initialised CTC head + severity classifier (router).

- **Base model**: `b2_wavlm_ctc_final` (fine-tuned WavLM-base, ALL weights frozen)
- **LoRA targets**: `q_proj`, `k_proj`, and `v_proj` in all 12 transformer layers (full attention low-rank adaptation), rank-16, α=40, 4 adapters
- **Adapters**: control (0), mild (1), moderate (2), severe (3)
- **Routing at train time**: ground-truth severity label selects adapter
- **Routing at inference time**: severity classifier (masked mean-pool → LayerNorm → GELU MLP)
- **Data split**: identical to B2 baseline (same speakers, seeds, fractions)

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"]      = "0"
os.environ["HF_HOME"]                   = "/kaggle/temp/hf_home"
os.environ["TRANSFORMERS_CACHE"]        = "/kaggle/temp/hf_home/models"
os.environ["HF_DATASETS_CACHE"]         = "/kaggle/temp/hf_home/datasets"
os.environ["TOKENIZERS_PARALLELISM"]    = "false"

os.makedirs("/kaggle/temp/hf_home/models",   exist_ok=True)
os.makedirs("/kaggle/temp/hf_home/datasets", exist_ok=True)
os.makedirs("/kaggle/temp/tb_p1_logs",       exist_ok=True)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
!pip -q install -U transformers datasets accelerate evaluate jiwer soundfile packaging

In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import evaluate
import random
import json
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union
from itertools import groupby

from datasets import load_dataset, DatasetDict
from transformers import (
    WavLMForCTC,
    Wav2Vec2Processor,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
)

## Step 1 — Load TORGO and Build the Split

**Identical** to B2 baseline: same `TEST_SPEAKERS`, `VAL_FRAC_PER_SPEAKER`, seeds, and control downsampling.

In [ ]:
# ── 1a) Load raw dataset ──
ds = load_dataset("abnerh/TORGO-database", cache_dir=os.environ["HF_DATASETS_CACHE"])
df = ds["train"].to_pandas()

df["audio_path"] = df["audio"].apply(lambda x: x["path"])
df["speaker"]    = df["audio_path"].apply(lambda p: str(p).split("_")[0])

print("Total utterances:", len(df))
print("Unique speakers:", sorted(df["speaker"].unique()))

In [ ]:
# ── 1b) Severity mapping ──
severity_mapping = {
    "F01": "severe",
    "M01": "severe",
    "M02": "severe",
    "M04": "severe",
    "M05": "moderate",
    "F03": "moderate",
    "F04": "mild",
    "M03": "mild",
    "FC01": "control",
    "FC02": "control",
    "FC03": "control",
    "MC01": "control",
    "MC02": "control",
    "MC03": "control",
    "MC04": "control",
}

df["Category"] = df["speaker"].map(severity_mapping)
print(df["Category"].value_counts())

In [ ]:
# ── 1c) Assign TEST speakers, then speaker-mixed validation split ──
TEST_SPEAKERS        = {"F01", "F04", "FC01", "M05"}
VAL_FRAC_PER_SPEAKER = 0.10
VAL_MAX_PER_SPEAKER  = 75
VAL_EXCLUDE_CATS     = {"mild"}
RANDOM_SEED          = 42

df["split"] = "train"
df.loc[df["speaker"].isin(TEST_SPEAKERS), "split"] = "test"

train_pool     = df[df["split"] == "train"].copy()
val_candidates = train_pool[~train_pool["Category"].isin(VAL_EXCLUDE_CATS)].copy()

val_parts = []
for speaker, spk_df in val_candidates.groupby("speaker"):
    n_val = max(1, int(round(len(spk_df) * VAL_FRAC_PER_SPEAKER)))
    n_val = min(n_val, VAL_MAX_PER_SPEAKER, len(spk_df))
    val_parts.append(spk_df.sample(n=n_val, random_state=RANDOM_SEED))

val_df = pd.concat(val_parts).sort_index().copy()
df.loc[val_df.index, "split"] = "val"

print("\nSplit counts:")
print(df["split"].value_counts())
print("\nSplit × Category breakdown:")
print(pd.crosstab(df["split"], df["Category"]))
print("\nValidation speakers:")
print(sorted(df.loc[df["split"] == "val", "speaker"].unique()))

In [ ]:
# ── 1d) Downsample control utterances in TRAINING only ──
CONTROL_TRAIN_TARGET = 1500

train_df    = df[df["split"] == "train"].copy()
train_control  = train_df[train_df["Category"] == "control"]
train_dysarth  = train_df[train_df["Category"] != "control"]

train_control_sampled = train_control.sample(
    n=min(CONTROL_TRAIN_TARGET, len(train_control)),
    random_state=RANDOM_SEED
)

train_df_final = pd.concat([train_dysarth, train_control_sampled]).sample(
    frac=1, random_state=RANDOM_SEED
)

val_df  = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print("Final training split:")
print(train_df_final["Category"].value_counts())
print(f"\nTotal train: {len(train_df_final)}")
print(f"Total val:   {len(val_df)}")
print(f"Total test:  {len(test_df)}")
print("\nTraining speakers:",   sorted(train_df_final["speaker"].unique()))
print("Validation speakers:", sorted(val_df["speaker"].unique()))
print("Test speakers:",       sorted(test_df["speaker"].unique()))

In [ ]:
# ── 1e) Convert back to HuggingFace DatasetDict ──
hf_full = ds["train"].add_column("speaker",  df["speaker"].tolist())
hf_full = hf_full.add_column("Category",     df["Category"].tolist())
hf_full = hf_full.add_column("split",        df["split"].tolist())

train_indices = train_df_final.index.tolist()
val_indices   = val_df.index.tolist()
test_indices  = test_df.index.tolist()

assert len(set(train_indices) & set(val_indices))  == 0
assert len(set(train_indices) & set(test_indices)) == 0
assert len(set(val_indices)   & set(test_indices)) == 0

hf_dict = DatasetDict({
    "train": hf_full.select(train_indices),
    "val":   hf_full.select(val_indices),
    "test":  hf_full.select(test_indices),
})

print(hf_dict)

## Step 2 — Add Severity Index & Load B2 Processor

The four severity adapters map to integer indices:
| Index | Adapter | Speakers |
|-------|---------|----------|
| 0 | control | FC01-FC03, MC01-MC04 |
| 1 | mild    | F04, M03 |
| 2 | moderate| F03, M05 |
| 3 | severe  | F01, M01, M02, M04 |

In [ ]:
SEVERITY_TO_IDX = {"control": 0, "mild": 1, "moderate": 2, "severe": 3}
ADAPTER_NAMES   = ["control", "mild", "moderate", "severe"]
N_ADAPTERS      = 4

def add_severity_idx(example):
    example["severity_idx"] = SEVERITY_TO_IDX.get(example.get("Category", "control"), 0)
    return example

hf_dict = hf_dict.map(add_severity_idx)
print("severity_idx added.")
print("Train severity distribution:",
      pd.Series(hf_dict["train"]["severity_idx"]).value_counts().sort_index().to_dict())

In [ ]:
import os

# ── Verify and Load processor from B2 checkpoint ──
B2_MODEL_PATH = "/kaggle/input/datasets/eshaniiparulekar/b2-model/kaggle/working/b2_wavlm_ctc_final"

# Confirm the path exists and show what's inside
if os.path.exists(B2_MODEL_PATH):
    print("✅ Path exists:", B2_MODEL_PATH)
    print("Contents:", os.listdir(B2_MODEL_PATH))
else:
    print("❌ Path NOT found. Searching for it...")
    # Walk up to find where the model actually landed
    base = "/kaggle/input/datasets/eshaniiparulekar/b2-model"
    for root, dirs, files in os.walk(base):
        if any(f in files for f in ["config.json", "vocab.json", "preprocessor_config.json"]):
            print("  Found candidate directory:", root)

processor = Wav2Vec2Processor.from_pretrained(B2_MODEL_PATH)
print("Loaded processor from:", B2_MODEL_PATH)
print("Vocab size:",            len(processor.tokenizer))
print("Pad token id:",          processor.tokenizer.pad_token_id)
print("Word delimiter:",        processor.tokenizer.word_delimiter_token)

## Step 3 — Preprocessing

Identical pipeline to B2: filter clips >15 s, normalise waveform, encode labels as character IDs.
`severity_idx` is preserved through the map.

In [ ]:
MAX_AUDIO_SAMPLES = 240_000  # 15 s at 16 kHz

def filter_long_audio(example):
    return len(example["audio"]["array"]) <= MAX_AUDIO_SAMPLES

before = {k: len(hf_dict[k]) for k in hf_dict}
hf_dict = hf_dict.filter(filter_long_audio, num_proc=1)
after  = {k: len(hf_dict[k]) for k in hf_dict}

print("Audio length filtering (max 15 s):")
for split in before:
    print(f"  {split}: {before[split]} -> {after[split]} ({before[split]-after[split]} removed)")

In [ ]:
def prepare_example(batch):
    audio = batch["audio"]
    proc_audio = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    )
    batch["input_values"]  = proc_audio["input_values"][0]
    batch["attention_mask"] = proc_audio["attention_mask"][0]
    text = batch["transcription"].lower().strip()
    batch["labels"] = processor.tokenizer(text).input_ids
    return batch

cols_to_remove = [
    c for c in hf_dict["train"].column_names
    if c not in ["audio", "transcription", "speaker", "Category", "split", "severity_idx"]
]

train_p = hf_dict["train"].map(prepare_example, remove_columns=cols_to_remove, num_proc=1)
val_p   = hf_dict["val"].map(prepare_example,   remove_columns=cols_to_remove, num_proc=1)
test_p  = hf_dict["test"].map(prepare_example,  remove_columns=cols_to_remove, num_proc=1)

print(train_p)
print(val_p)
print(test_p)

## Step 4 — Data Collator

Same as B2 with one addition: `severity_idx` is stacked into the batch as a `torch.long` tensor.

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = "longest"

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_values": f["input_values"], "attention_mask": f["attention_mask"]}
            for f in features
        ]
        label_features  = [{"input_ids": f["labels"]} for f in features]
        severity_indices = torch.tensor(
            [f["severity_idx"] for f in features], dtype=torch.long
        )

        batch = self.processor.feature_extractor.pad(
            input_features, padding=self.padding, return_tensors="pt"
        )
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding=self.padding, return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"]       = labels
        batch["severity_idx"] = severity_indices
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)
print("Data collator ready.")

## Step 5 — Multi-Adapter LoRA Module

Each `MultiAdapterLoRALinear` wraps a frozen `nn.Linear` and holds **N separate LoRA (A, B) pairs** —
one per severity level.  The active adapter is selected by setting `_active_adapter` (int) before
calling forward; `None` returns the frozen base output unchanged.

In [ ]:
class MultiAdapterLoRALinear(nn.Module):
    """Frozen linear layer with N severity-specific LoRA adapters.

    Active adapter is set via `_active_adapter` (int or None). Each `lora_B` starts at
    zero so adapters begin as identity; `lora_A` uses small Gaussian init (σ = 1/√r).
    """
    def __init__(
        self,
        base_linear: nn.Linear,
        n_adapters: int = 4,
        r: int = 16,
        lora_alpha: int = 32,
        dropout: float = 0.05,
    ):
        super().__init__()
        self.base        = base_linear
        self.r           = r
        self.scaling     = lora_alpha / r
        self.n_adapters  = n_adapters
        in_f  = base_linear.in_features
        out_f = base_linear.out_features
        # Freeze the original weights
        for p in self.base.parameters():
            p.requires_grad = False
        # Per-adapter LoRA matrices
        self.lora_A = nn.ParameterList([
            nn.Parameter(torch.empty(r, in_f)) for _ in range(n_adapters)
        ])
        self.lora_B = nn.ParameterList([
            nn.Parameter(torch.zeros(out_f, r)) for _ in range(n_adapters)
        ])
        # Gaussian A / zero B (LoRA-style); std scales with rank for stable early training
        for A in self.lora_A:
            nn.init.normal_(A, mean=0.0, std=1.0 / math.sqrt(r))
        self.lora_dropout        = nn.Dropout(dropout)
        self._active_adapter: Optional[int] = None

    # ── Proxy any unknown attribute lookups to the base linear layer ──
    def __getattr__(self, name: str):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.base, name)

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base(x)
        if self._active_adapter is None:
            return base_out
        idx   = self._active_adapter
        delta = self.lora_dropout(x) @ self.lora_A[idx].t()   # (..., r)
        delta = delta @ self.lora_B[idx].t()                    # (..., out_f)
        return base_out + delta * self.scaling

print("MultiAdapterLoRALinear defined.")

## Step 6 — Build the Model

1. Load `b2_wavlm_ctc_final` (the TORGO-fine-tuned WavLM-base)  
2. Freeze **all** parameters (encoder + original CTC head)  
3. Inject `MultiAdapterLoRALinear` into `q_proj`, `k_proj`, and `v_proj` in all 12 encoder layers  
4. Re-initialise the CTC head and make it trainable  
5. Wrap in `SeverityRoutedWavLM` which exposes `set_adapter(idx)`

In [ ]:
LORA_R       = 16
LORA_ALPHA   = 40   # α/r = 2.5 — slightly stronger updates with q/k/v adapters
LORA_DROPOUT = 0.03

# ── Load B2 fine-tuned model ──
base_model = WavLMForCTC.from_pretrained(
    B2_MODEL_PATH,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    ctc_zero_infinity=True,
)

# ── Freeze all parameters ──
for param in base_model.parameters():
    param.requires_grad = False

# ── Inject LoRA into q_proj, k_proj, v_proj (all 12 encoder layers) ──
lora_modules: List[MultiAdapterLoRALinear] = []

for layer_idx, layer in enumerate(base_model.wavlm.encoder.layers):
    attn = layer.attention

    for proj_name in ("q_proj", "k_proj", "v_proj"):
        base_linear = getattr(attn, proj_name)
        lora = MultiAdapterLoRALinear(
            base_linear,
            n_adapters=N_ADAPTERS,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            dropout=LORA_DROPOUT,
        )
        setattr(attn, proj_name, lora)
        lora_modules.append(lora)

print(f"Injected {len(lora_modules)} LoRA modules (q_proj + k_proj + v_proj × 12 layers).")

# ── Re-initialise CTC head and make it trainable ──
hidden_size = base_model.config.hidden_size  # 768
vocab_size  = len(processor.tokenizer)

base_model.lm_head = nn.Linear(hidden_size, vocab_size)
nn.init.normal_(base_model.lm_head.weight, std=0.02)
nn.init.zeros_(base_model.lm_head.bias)
for p in base_model.lm_head.parameters():
    p.requires_grad = True

# ── Parameter count ──
total     = sum(p.numel() for p in base_model.parameters())
trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)

print(f"\nTotal parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"  of which LoRA:      {sum(p.numel() for m in lora_modules for p in m.parameters() if p.requires_grad):,}")
print(f"  of which CTC head:  {sum(p.numel() for p in base_model.lm_head.parameters()):,}")
print(f"Frozen parameters:    {total - trainable:,}")

In [ ]:
class SeverityRoutedWavLM(nn.Module):
    """WavLMForCTC with severity-specific LoRA routing.
    
    call set_adapter(sev_idx) before each forward to activate the correct adapter.
    set_adapter(None) disables all adapters (pure frozen base).
    """
    def __init__(self, base_model: WavLMForCTC, lora_modules: List[MultiAdapterLoRALinear]):
        super().__init__()
        self.model = base_model
        # Plain list — NOT nn.ModuleList — avoids double-registration
        # The modules are already reachable via self.model.wavlm.encoder.layers...
        self._lora_refs: List[MultiAdapterLoRALinear] = lora_modules

    def set_adapter(self, adapter_idx: Optional[int]):
        for m in self._lora_refs:         # iterate the plain list
            m._active_adapter = adapter_idx

    # rest of the class unchanged...

    def forward(
        self,
        input_values: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        **kwargs,
    ):
        """Delegates to base model; adapter must be set externally."""
        return self.model(
            input_values=input_values,
            attention_mask=attention_mask,
            labels=labels,
        )

    @property
    def config(self):
        return self.model.config


model = SeverityRoutedWavLM(base_model, lora_modules)
model.gradient_checkpointing_enable = base_model.gradient_checkpointing_enable
model.gradient_checkpointing_enable()
print("SeverityRoutedWavLM ready.")

## Step 7 — Metrics & Callbacks

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def decode_ctc_predictions(pred_ids, tokenizer):
    blank_id = tokenizer.pad_token_id
    decoded  = []
    for seq in pred_ids:
        collapsed = [k for k, _ in groupby(seq)]
        filtered  = [t for t in collapsed if t != blank_id]
        decoded.append(
            tokenizer.decode(filtered, skip_special_tokens=True) if filtered else ""
        )
    return decoded


def compute_metrics(pred):
    pred_ids   = np.argmax(pred.predictions, axis=-1)
    label_ids  = pred.label_ids
    label_ids  = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)

    pred_str  = decode_ctc_predictions(pred_ids, processor.tokenizer)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str  = [p.strip() for p in pred_str]
    label_str = [l.strip() for l in label_str]

    pred_eval  = [p if p else "⟨empty⟩" for p in pred_str]
    label_eval = [l if l else "⟨empty⟩" for l in label_str]

    return {
        "wer": wer_metric.compute(predictions=pred_eval, references=label_eval),
        "cer": cer_metric.compute(predictions=pred_eval, references=label_eval),
    }


print("Metrics ready.")

In [ ]:
class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if "loss" in logs:
            print(f"[step {state.global_step}] train_loss={logs['loss']:.4f}")
        if "eval_cer" in logs:
            print(f"[step {state.global_step}] val_cer={logs['eval_cer']:.4f}  val_wer={logs.get('eval_wer', 'N/A')}")


class SanityCheckCallback(TrainerCallback):
    """Decodes one val sample after each eval to verify the model is learning."""
    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        model.eval()
        device = next(model.parameters()).device
        try:
            sample       = val_p[0]
            sev_idx      = sample["severity_idx"]
            input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)
            attn_mask    = torch.tensor(sample["attention_mask"]).unsqueeze(0).to(device)

            with torch.no_grad():
                model.set_adapter(sev_idx)
                logits = model(
                    input_values=input_values,
                    attention_mask=attn_mask,
                ).logits
                model.set_adapter(None)

            pred_ids  = torch.argmax(logits, dim=-1).cpu().numpy()
            pred_text = decode_ctc_predictions(pred_ids, processor.tokenizer)[0]
            label_ids = [x for x in sample["labels"] if x != -100]
            ref_text  = processor.tokenizer.decode(label_ids, skip_special_tokens=True)

            print(f"  [SANITY] REF : {ref_text}")
            print(f"  [SANITY] PRED: {pred_text}")
        except Exception as e:
            print(f"  [SANITY] Error: {e}")


early_stopping = EarlyStoppingCallback(
    early_stopping_patience=8,
    early_stopping_threshold=0.001,
)

print("Callbacks ready.")

## Step 8 — Custom Trainer with Severity Routing

`SeverityRoutingTrainer` overrides two methods:

- **`compute_loss`**: iterates over the 4 severity groups present in each mini-batch,
  activates the matching adapter, accumulates weighted CTC loss.
- **`prediction_step`**: same routing logic but collects logits for metric computation.
  Because all samples in a collated batch share the same padded input length,
  encoder outputs across severity groups have the same time dimension `T` and can be
  stacked into a single `(batch, T, V)` tensor.

In [ ]:
class SeverityRoutingTrainer(Trainer):

    # ── Training ──────────────────────────────────────────────────────────
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        severity_idx  = inputs.pop("severity_idx", None)
        input_values  = inputs["input_values"]
        attention_mask = inputs.get("attention_mask")
        labels        = inputs.get("labels")

        if severity_idx is None:
            # Fallback: no routing (shouldn't occur in normal training)
            model.set_adapter(None)
            outputs = model(input_values=input_values,
                            attention_mask=attention_mask,
                            labels=labels)
            return (outputs.loss, outputs) if return_outputs else outputs.loss

        loss_parts: List[torch.Tensor] = []
        last_outputs = None

        for sev in range(N_ADAPTERS):
            mask = (severity_idx == sev)
            if not mask.any():
                continue
            model.set_adapter(sev)
            outs = model(
                input_values=input_values[mask],
                attention_mask=attention_mask[mask] if attention_mask is not None else None,
                labels=labels[mask] if labels is not None else None,
            )
            if outs.loss is not None:
                loss_parts.append(outs.loss * mask.sum().float())
            last_outputs = outs

        model.set_adapter(None)

        total_loss = torch.stack(loss_parts).sum() / len(severity_idx)

        return (total_loss, last_outputs) if return_outputs else total_loss

    # ── Evaluation ────────────────────────────────────────────────────────
    def prediction_step(
        self,
        model: nn.Module,
        inputs: Dict[str, Any],
        prediction_loss_only: bool,
        ignore_keys: Optional[List[str]] = None,
    ) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor], Optional[torch.Tensor]]:

        severity_idx   = inputs.pop("severity_idx", None)
        input_values   = inputs["input_values"]
        attention_mask = inputs.get("attention_mask")
        labels         = inputs.get("labels")
        batch_size     = input_values.shape[0]

        with torch.no_grad():
            if severity_idx is not None:
                full_logits: Optional[torch.Tensor] = None
                total_loss  = 0.0
                total_n     = 0

                for sev in range(N_ADAPTERS):
                    mask = (severity_idx == sev)
                    if not mask.any():
                        continue
                    model.set_adapter(sev)
                    outs = model(
                        input_values=input_values[mask],
                        attention_mask=attention_mask[mask] if attention_mask is not None else None,
                        labels=labels[mask] if labels is not None else None,
                    )
                    # All samples share the same padded T dim — safe to index by mask
                    if full_logits is None:
                        T, V = outs.logits.shape[1], outs.logits.shape[2]
                        full_logits = torch.zeros(
                            batch_size, T, V, device=input_values.device
                        )
                    full_logits[mask] = outs.logits
                    if outs.loss is not None:
                        n = mask.sum().item()
                        total_loss += outs.loss.item() * n
                        total_n    += n

                model.set_adapter(None)
                loss = torch.tensor(total_loss / total_n if total_n > 0 else 0.0)

            else:
                model.set_adapter(None)
                outs        = model(input_values=input_values,
                                    attention_mask=attention_mask,
                                    labels=labels)
                full_logits = outs.logits
                loss        = outs.loss.detach() if outs.loss is not None else None

        if prediction_loss_only:
            return (loss, None, None)
        return (loss, full_logits, labels)


print("SeverityRoutingTrainer defined.")

## Step 9 — Training Arguments & Train

In [ ]:
CHECKPOINT_DIR = "/kaggle/working/p1_severity_lora"

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    # Training schedule
    num_train_epochs=30,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch = 16 (matches B2)
    learning_rate=1e-3,                 # higher LR: adapters + CTC head are new
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    weight_decay=0.01,

    # Eval + checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,

    # Memory + stability
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    max_grad_norm=1.0,

    # Logging
    logging_steps=25,
    logging_dir="/kaggle/temp/tb_p1_logs",
    report_to="none",

    # Keep severity_idx and other extra columns in batches
    remove_unused_columns=False,
    save_total_limit=3,
)

print("Training arguments set.")
print(f"Effective batch size: "
      f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

In [ ]:
trainer = SeverityRoutingTrainer(
    model=model,
    args=training_args,
    train_dataset=train_p,
    eval_dataset=val_p,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[PrintLossCallback(), SanityCheckCallback(), early_stopping],
)

print("\n--- DISK USAGE BEFORE TRAINING ---")
!du -sh /kaggle/working/* 2>/dev/null || true

print("\nStarting training...")
trainer.train()

## Step 10 — Severity Classifier

Trained **after** CTC on frozen encoder outputs. Uses **masked mean pooling** (padding ignored), **LayerNorm** on the pooled vector, and a **two-block GELU MLP** with **class-weighted** cross-entropy and **label smoothing** for imbalanced severity labels.

Cached pooled vectors keep secondary training fast.

In [ ]:
def masked_mean_pool(
    hidden_states: torch.Tensor,
    attention_mask: Optional[torch.Tensor],
) -> torch.Tensor:
    """Mean over time with padding masked out (feature_extractor attention_mask)."""
    if attention_mask is None:
        return hidden_states.mean(dim=1)
    m = attention_mask.unsqueeze(-1).to(dtype=hidden_states.dtype)
    summed = (hidden_states * m).sum(dim=1)
    denom = m.sum(dim=1).clamp(min=1e-9)
    return summed / denom


class SeverityClassifier(nn.Module):
    """Pooled-utterance severity head: LayerNorm + wide GELU MLP."""

    def __init__(
        self,
        hidden_size: int = 768,
        n_classes: int = 4,
        dropout: float = 0.15,
        intermediate: int = 384,
    ):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_size)
        hid2 = max(n_classes * 4, intermediate // 2)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, intermediate),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(intermediate, hid2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hid2, n_classes),
        )

    def forward_pooled(self, pooled: torch.Tensor) -> torch.Tensor:
        """Classify already pooled (B, hidden) vectors — used for cached training."""
        return self.mlp(self.norm(pooled))

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        pooled = masked_mean_pool(hidden_states, attention_mask)
        return self.forward_pooled(pooled)


print("SeverityClassifier defined.")

In [ ]:
from torch.utils.data import DataLoader as TorchDataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

severity_clf = SeverityClassifier(hidden_size=768, n_classes=N_ADAPTERS).to(device)

# ── Extract frozen encoder representations for train + val ──
# We only need mean-pooled hidden states — no gradient needed.
print("Extracting encoder representations for classifier training...")


def extract_representations(dataset, batch_size=24):
    """Return (masked mean-pooled encoder vectors [N, 768], severity labels [N])."""
    loader = TorchDataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=data_collator,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    all_reps, all_labels = [], []
    model.set_adapter(None)

    with torch.no_grad():
        for batch in loader:
            iv   = batch["input_values"].to(device, non_blocking=True)
            mask = batch.get("attention_mask")
            if mask is not None:
                mask = mask.to(device, non_blocking=True)
            sev  = batch["severity_idx"]

            hidden = model.model.wavlm(
                input_values=iv,
                attention_mask=mask,
            ).last_hidden_state

            pooled = masked_mean_pool(hidden, mask).cpu()
            all_reps.append(pooled)
            all_labels.append(sev)

    return torch.cat(all_reps), torch.cat(all_labels)


train_reps,  train_sev_labels = extract_representations(train_p)
val_reps,    val_sev_labels   = extract_representations(val_p)

print(f"Train repr: {train_reps.shape}  labels: {train_sev_labels.shape}")
print(f"Val   repr: {val_reps.shape}    labels: {val_sev_labels.shape}")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader as TDL

CLF_EPOCHS = 40
clf_train_loader = TDL(
    TensorDataset(train_reps, train_sev_labels),
    batch_size=128,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
)
clf_val_loader = TDL(
    TensorDataset(val_reps, val_sev_labels),
    batch_size=256,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
)

# Inverse-frequency class weights (mean-normalised) + mild label smoothing
_counts = torch.bincount(train_sev_labels.long(), minlength=N_ADAPTERS).float()
_class_w = (_counts.sum() / _counts.clamp(min=1.0)).to(device)
_class_w = _class_w / _class_w.mean()

clf_optim = AdamW(
    severity_clf.parameters(),
    lr=3e-4,
    weight_decay=0.02,
    betas=(0.9, 0.98),
)
clf_scheduler = CosineAnnealingLR(clf_optim, T_max=CLF_EPOCHS)
ce_loss = nn.CrossEntropyLoss(weight=_class_w, label_smoothing=0.08)

print(f"Training severity classifier for {CLF_EPOCHS} epochs...")
best_clf_acc = 0.0
patience, bad_epochs = 10, 0

for epoch in range(1, CLF_EPOCHS + 1):
    severity_clf.train()
    epoch_loss = 0.0
    for reps, lbl in clf_train_loader:
        reps = reps.to(device, non_blocking=True)
        lbl = lbl.to(device, non_blocking=True)
        clf_optim.zero_grad()
        logits = severity_clf.forward_pooled(reps)
        loss = ce_loss(logits, lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(severity_clf.parameters(), 1.0)
        clf_optim.step()
        epoch_loss += loss.item() * len(lbl)

    clf_scheduler.step()

    severity_clf.eval()
    correct = total = 0
    with torch.no_grad():
        for reps, lbl in clf_val_loader:
            reps = reps.to(device, non_blocking=True)
            lbl = lbl.to(device, non_blocking=True)
            preds = severity_clf.forward_pooled(reps).argmax(dim=-1)
            correct += (preds == lbl).sum().item()
            total += len(lbl)
    acc = correct / total
    if acc > best_clf_acc:
        best_clf_acc = acc
        bad_epochs = 0
        torch.save(severity_clf.state_dict(), "/kaggle/working/p1_severity_classifier.pt")
    else:
        bad_epochs += 1

    if epoch % 5 == 0 or epoch == 1:
        print(
            f"  Epoch {epoch:2d}  loss={epoch_loss/len(train_reps):.4f}  "
            f"val_acc={acc:.4f}  lr={clf_optim.param_groups[0]['lr']:.2e}"
        )
    if bad_epochs >= patience:
        print(f"  Early stop at epoch {epoch} (no val acc improvement for {patience} epochs).")
        break

print(f"\nBest classifier val accuracy: {best_clf_acc:.4f}")
severity_clf.load_state_dict(torch.load("/kaggle/working/p1_severity_classifier.pt"))
print("Best classifier weights restored.")

## Step 11 — Evaluate on Test Set

Two evaluation modes:
1. **Oracle routing** — ground-truth severity used to select adapter (upper bound)
2. **Predicted routing** — severity classifier selects adapter (realistic inference)

In [ ]:
from torch.utils.data import DataLoader as EvalDL


def evaluate_split(
    dataset,
    routing: str = "oracle",   # "oracle" | "predicted"
    batch_size: int = 8,
):
    """Compute WER/CER on dataset.
    routing='oracle'    → use ground-truth severity_idx
    routing='predicted' → classify severity from encoder output, then transcribe
    """
    loader = EvalDL(
        dataset,
        batch_size=batch_size,
        collate_fn=data_collator,
        shuffle=False,
        num_workers=2,
    )
    model.eval()
    severity_clf.eval()

    all_pred, all_ref = [], []

    with torch.no_grad():
        for batch in loader:
            iv          = batch["input_values"].to(device)
            attn_mask   = batch.get("attention_mask")
            if attn_mask is not None:
                attn_mask = attn_mask.to(device)
            labels      = batch["labels"]
            gt_sev      = batch["severity_idx"].to(device)
            batch_size_ = iv.shape[0]

            if routing == "oracle":
                sev_used = gt_sev
            else:
                model.set_adapter(None)
                hidden = model.model.wavlm(
                    input_values=iv, attention_mask=attn_mask
                ).last_hidden_state
                pooled = masked_mean_pool(hidden, attn_mask)
                sev_used = severity_clf.forward_pooled(pooled).argmax(dim=-1)

            # Run transcription with routed adapters
            full_logits = None
            for sev in range(N_ADAPTERS):
                mask = (sev_used == sev)
                if not mask.any():
                    continue
                model.set_adapter(sev)
                outs = model(
                    input_values=iv[mask],
                    attention_mask=attn_mask[mask] if attn_mask is not None else None,
                )
                if full_logits is None:
                    T, V = outs.logits.shape[1], outs.logits.shape[2]
                    full_logits = torch.zeros(batch_size_, T, V, device=iv.device)
                full_logits[mask] = outs.logits
            model.set_adapter(None)

            pred_ids  = torch.argmax(full_logits, dim=-1).cpu().numpy()
            pred_str  = decode_ctc_predictions(pred_ids, processor.tokenizer)

            label_ids = labels.numpy()
            label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)
            label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

            all_pred.extend([p.strip() for p in pred_str])
            all_ref.extend([l.strip()  for l in label_str])

    # Avoid empty-string divide by zero in metrics
    pred_eval  = [p if p else "⟨empty⟩" for p in all_pred]
    label_eval = [l if l else "⟨empty⟩" for l in all_ref]

    wer = wer_metric.compute(predictions=pred_eval, references=label_eval)
    cer = cer_metric.compute(predictions=pred_eval, references=label_eval)
    return {"wer": wer, "cer": cer, "n_samples": len(all_pred)}


print("Evaluating on TEST set (oracle routing)...")
test_oracle = evaluate_split(test_p, routing="oracle")
print(f"  WER (oracle): {test_oracle['wer']:.4f}   CER (oracle): {test_oracle['cer']:.4f}")

print("Evaluating on TEST set (predicted routing)...")
test_predicted = evaluate_split(test_p, routing="predicted")
print(f"  WER (pred):   {test_predicted['wer']:.4f}   CER (pred):   {test_predicted['cer']:.4f}")

print("\nEvaluating on VAL set (oracle routing)...")
val_oracle = evaluate_split(val_p, routing="oracle")
print(f"  WER (oracle): {val_oracle['wer']:.4f}   CER (oracle): {val_oracle['cer']:.4f}")

## Per-severity breakdown

The next code cell decodes the test set with **oracle** routing, prints sample lines, then writes `p1_test_results.csv` and `p1_summary.json`.

In [ ]:
all_preds, all_labels, all_cats = [], [], []

model.eval()
loader = EvalDL(test_p, batch_size=8, collate_fn=data_collator, shuffle=False, num_workers=2)

with torch.no_grad():
    for batch in loader:
        iv        = batch["input_values"].to(device)
        attn_mask = batch.get("attention_mask")
        if attn_mask is not None:
            attn_mask = attn_mask.to(device)
        gt_sev    = batch["severity_idx"].to(device)
        labels    = batch["labels"]

        full_logits = None
        for sev in range(N_ADAPTERS):
            mask = (gt_sev == sev)
            if not mask.any():
                continue
            model.set_adapter(sev)
            outs = model(input_values=iv[mask],
                         attention_mask=attn_mask[mask] if attn_mask is not None else None)
            if full_logits is None:
                T, V = outs.logits.shape[1], outs.logits.shape[2]
                full_logits = torch.zeros(iv.shape[0], T, V, device=device)
            full_logits[mask] = outs.logits
        model.set_adapter(None)

        pred_ids  = torch.argmax(full_logits, dim=-1).cpu().numpy()
        pred_str  = decode_ctc_predictions(pred_ids, processor.tokenizer)
        label_ids = labels.numpy()
        label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)
        label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        all_preds.extend([p.strip() for p in pred_str])
        all_labels.extend([l.strip() for l in label_str])
        all_cats.extend([ADAPTER_NAMES[s] for s in batch["severity_idx"].tolist()])

# ── Sample predictions ──
print("\nSample predictions (first 15 test utterances):")
print("-" * 70)
for i in range(min(15, len(all_preds))):
    print(f"[{all_cats[i]:10s}] REF : {all_labels[i]}")
    print(f"           PRED: {all_preds[i]}")
    print()

# ── Per-severity breakdown & files ──
results_df = pd.DataFrame({
    "prediction": all_preds,
    "reference":  all_labels,
    "Category":   all_cats,
})

print("Per-severity results:")
print("-" * 50)

for cat in ["control", "mild", "moderate", "severe"]:
    subset = results_df[results_df["Category"] == cat]
    subset = subset[subset["reference"].str.strip().str.len() > 0]
    if len(subset) == 0:
        print(f"{cat:15s}: no samples")
        continue

    preds = [p if len(p) > 0 else "⟨empty⟩" for p in subset["prediction"].tolist()]
    cat_wer = wer_metric.compute(predictions=preds, references=subset["reference"].tolist())
    cat_cer = cer_metric.compute(predictions=preds, references=subset["reference"].tolist())
    print(f"{cat:15s}: WER={cat_wer*100:.2f}%  CER={cat_cer*100:.2f}%  (n={len(subset)})")

print()

results_df.to_csv("/kaggle/working/p1_test_results.csv", index=False)
print("Results saved to /kaggle/working/p1_test_results.csv")

summary = {
    "model":               "P1_SeverityRouted_LoRA",
    "base_model":          B2_MODEL_PATH,
    "lora_r":              LORA_R,
    "lora_alpha":          LORA_ALPHA,
    "lora_targets":        "q_proj,k_proj,v_proj",
    "n_adapters":          N_ADAPTERS,
    "adapter_names":       ADAPTER_NAMES,
    "overall_wer_oracle":  round(test_oracle["wer"], 4),
    "overall_cer_oracle":  round(test_oracle["cer"], 4),
    "test_speakers":       ["F01", "F04", "FC01", "M05"],
}
with open("/kaggle/working/p1_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Summary saved to /kaggle/working/p1_summary.json")

## Step 12 — Save

In [ ]:
FINAL_MODEL_PATH = "/kaggle/working/p1_severity_lora_final"

# Save base model weights (includes re-initialised CTC head and injected LoRA layers)
trainer.save_model(FINAL_MODEL_PATH)
processor.save_pretrained(FINAL_MODEL_PATH)

# Save LoRA adapter weights separately for easy re-loading
lora_state = {}
for i, m in enumerate(lora_modules):
    lora_state[f"module_{i}_lora_A"] = [A.data for A in m.lora_A]
    lora_state[f"module_{i}_lora_B"] = [B.data for B in m.lora_B]
torch.save(lora_state, os.path.join(FINAL_MODEL_PATH, "lora_adapters.pt"))

# Save severity classifier
torch.save(severity_clf.state_dict(), os.path.join(FINAL_MODEL_PATH, "severity_classifier.pt"))

# Save results summary
results = {
    "model": "P1 Severity-Stratified LoRA Routing",
    "base": B2_MODEL_PATH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_targets": "q_proj,k_proj,v_proj",
    "n_adapters": N_ADAPTERS,
    "adapter_names": ADAPTER_NAMES,
    "test_wer_oracle":    test_oracle["wer"],
    "test_cer_oracle":    test_oracle["cer"],
    "test_wer_predicted": test_predicted["wer"],
    "test_cer_predicted": test_predicted["cer"],
    "val_wer_oracle":     val_oracle["wer"],
    "val_cer_oracle":     val_oracle["cer"],
}
with open(os.path.join(FINAL_MODEL_PATH, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

print("Saved to:", FINAL_MODEL_PATH)
print(json.dumps(results, indent=2))

print("\n--- DISK USAGE AFTER TRAINING ---")
!du -sh /kaggle/working/* 2>/dev/null || true